# AI/ML Internship Task – Week 01
## Task 1: Image Classification Model

**Objective:** Develop a deep learning model capable of accurately classifying images into predefined categories using a Convolutional Neural Network (CNN).

**Dataset:** MNIST — 70,000 28x28 grayscale images of handwritten digits (0-9)

**Framework:** TensorFlow / Keras


## Step 1 & 2: Dataset Selection and Preprocessing

In [ ]:
import gzip
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = [str(i) for i in range(10)]
print("TensorFlow version:", tf.__version__)

In [ ]:
def _read_idx_images(path):
    with gzip.open(path, "rb") as f:
        data = f.read()
    num_images = int.from_bytes(data[4:8], "big")
    rows = int.from_bytes(data[8:12], "big")
    cols = int.from_bytes(data[12:16], "big")
    images = np.frombuffer(data, dtype=np.uint8, offset=16)
    return images.reshape(num_images, rows, cols)

def _read_idx_labels(path):
    with gzip.open(path, "rb") as f:
        data = f.read()
    return np.frombuffer(data, dtype=np.uint8, offset=8)

x_train_full = _read_idx_images("data/train-images-idx3-ubyte.gz")
y_train_full = _read_idx_labels("data/train-labels-idx1-ubyte.gz")
x_test = _read_idx_images("data/t10k-images-idx3-ubyte.gz")
y_test = _read_idx_labels("data/t10k-labels-idx1-ubyte.gz")

print("Train images:", x_train_full.shape)
print("Test images:", x_test.shape)

### Visualize a few sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_full[i], cmap="gray")
    ax.set_title(f"Label: {y_train_full[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Normalize pixel values and split into train / validation / test sets

In [ ]:
# Normalize pixel values to [0, 1] and add channel dimension
x_train_full = x_train_full.astype("float32")[..., np.newaxis] / 255.0
x_test = x_test.astype("float32")[..., np.newaxis] / 255.0
y_train_full = y_train_full.astype("int64")
y_test = y_test.astype("int64")

# Split training data into train/validation sets (90/10)
val_split = int(0.9 * len(x_train_full))
x_train, x_val = x_train_full[:val_split], x_train_full[val_split:]
y_train, y_val = y_train_full[:val_split], y_train_full[val_split:]

print(f"Training samples:   {x_train.shape[0]}")
print(f"Validation samples: {x_val.shape[0]}")
print(f"Test samples:       {x_test.shape[0]}")

## Step 3: Model Development (CNN Architecture)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # Convolutional Block 1
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Convolutional Block 2
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Fully Connected Layers
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
], name="MNIST_CNN")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## Step 4: Model Training

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=8,
    batch_size=256,
    callbacks=callbacks,
    verbose=2
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history["accuracy"], label="Training Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title("Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"], label="Training Loss")
axes[1].plot(history.history["val_loss"], label="Validation Loss")
axes[1].set_title("Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Model Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix - MNIST Test Set")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

## Step 6: Results Analysis

**Summary:**
- The CNN achieved **~99.0% test accuracy**, with macro-averaged precision, recall, and F1-score all around 0.99.
- The confusion matrix shows the model rarely confuses digits; the most common (small) confusions are between visually similar digits such as 4/9 and 3/5.
- Training and validation accuracy converge closely with no significant gap, indicating the model is not overfitting, aided by BatchNormalization and Dropout layers.

**Possible Improvements:**
- Add data augmentation (rotation, shift, zoom) to further improve robustness to handwriting style variation.
- Experiment with deeper architectures or residual connections for marginal accuracy gains.
- Apply learning rate warm-up or cosine decay scheduling.
- Try ensembling multiple CNN models to boost accuracy further.
- Evaluate on out-of-distribution handwritten samples to test generalization beyond MNIST's collection conditions.


## Save the Trained Model

In [ ]:
model.save("models/mnist_cnn_model.keras")
print("Model saved to models/mnist_cnn_model.keras")